<a href="https://colab.research.google.com/github/fvangool/Deep-Learning-Specialization-Coursera/blob/main/XGBoost_v22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install wandb -qU

In [ ]:
"""
XGBoost v22 — Central FE Parquet + W&B + All v21 Features
===========================================================
Changes vs v20:

  FEATURE ENGINEERING
  ───────────────────
  + Loads pre-engineered parquet from feature_engineering_v21.py
    (no local engineer_features() — single source of truth)
  + All v21 features included automatically via parquet:
      dual_boundary_proximity, boundary_proximity
      prior_irrigation_intensity, high_prior_irrigation
      humidity_at_boundary, vpd_at_boundary
      sandy_at_boundary, canal_at_boundary
      margin_Medium_wins, formula_predicts_medium (via local vars)
      dec2 features for all NUMS
  - score_High/Low/Medium REMOVED (zero variance on hard rows — v20 bug)
  - margin_High/Low_Medium REMOVED (redundant with magic_score)
  + MD5 checksum verification on loaded parquet
  + Zero-variance guard on hard rows runs from parquet metadata

  W&B (unchanged from v20 — all 4 points)
  ─────────────────────────────────────────
  + Artifact versioning: OOF/pred .npy + parquet dataset artifact
  + Fold + seed + global BA tracking (monotonic global_fold_num)
  + Hard-example confusion matrix (pre- and post-bias)
  + Config tracking: FE version, augmentation, hyperparams

  RAM FIXES (carried forward from v20)
  ─────────────────────────────────────
  + CAT×CAT + CAT×NUM pairwise interactions only (no NUM×NUM)
  + numpy hstack for DMatrix construction (no pd.concat in fold loop)
  + del train_eng/test_eng immediately after Section 5

  UNCHANGED FROM v20
  ──────────────────
  - Native xgb.train() API with feval_bal_acc
  - Optuna-tuned hyperparameters
  - Multi-seed (5 seeds × 5 folds)
  - Asymmetric pseudolabelling (post-bias, downweighted)
  - Bias tuning (logit-space coordinate descent)
  - Telegram notifier + heartbeat
  - Diagnostic plots

SAVE CONVENTION
───────────────
  oof_xgb_v22.npy         (n_competition, 3)  Low=0  Med=1  High=2
  pred_xgb_v22.npy        (n_test, 3)
  oof_xgb_v22_biased.npy  (alias)
  pred_xgb_v22_biased.npy (alias)
"""

# ============================================================
# SETUP (run once if needed)
# !pip install wandb -q
# ============================================================

# ============================================================
# IMPORTS
# ============================================================
import gc
import os
import json
import time
import random
import hashlib
import warnings
import threading
import traceback
import urllib.request
from contextlib import contextmanager
from itertools import combinations

os.environ["PYTHONHASHSEED"] = "42"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import wandb
from google.colab import userdata
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from scipy.special import logit
from tqdm import tqdm

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

# ============================================================
# SECTION 0 — CONFIGURATION
# ============================================================

# ── Paths ─────────────────────────────────────────────────────
OOF_DIR       = "/content/drive/MyDrive/irrigation_need_v15/"
SUB_DIR       = "/content/"
TEST_PATH     = f"{OOF_DIR}test.csv"
HARD_IDX_PATH = f"{OOF_DIR}hard_example_indices.npy"

# ── Central FE parquet (produced by feature_engineering_v21.py) ─
FE_VERSION      = "v21"
TRAIN_PARQUET   = f"{OOF_DIR}train_engineered_{FE_VERSION}.parquet"
TEST_PARQUET    = f"{OOF_DIR}test_engineered_{FE_VERSION}.parquet"
FE_META_PATH    = f"{OOF_DIR}feature_metadata_{FE_VERSION}.json"

# ── W&B ──────────────────────────────────────────────────────
WANDB_PROJECT  = "ps-s6e4-irrigation"
WANDB_ENTITY   = "wblackstone-twilight-signals"
WANDB_ENABLED  = True

# ── Run identity ─────────────────────────────────────────────
TAG      = "xgb_v22"
RUN_NAME = "XGB v22"

# ── Telegram ─────────────────────────────────────────────────
TELEGRAM_BOT_TOKEN = "8755783601:AAGuCUzM6CdgjA825tep5f5zj0FNd7iSkp4"
TELEGRAM_CHAT_ID   = "5422067007"
TELEGRAM_ENABLED   = True

# ── CV ───────────────────────────────────────────────────────
SEEDS   = [42, 123, 2024, 7, 314]
N_FOLDS = 5
N_CLASSES = 3

# ── Target ───────────────────────────────────────────────────
TARGET             = "Irrigation_Need"
TARGET_MAPPING     = {"Low": 0, "Medium": 1, "High": 2}
INV_TARGET_MAPPING = {0: "Low", 1: "Medium", 2: "High"}
CLASS_NAMES        = ["Low", "Medium", "High"]

# ── Raw feature column lists (also recovered from FE metadata) ─
NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon",
    "Electrical_Conductivity", "Temperature_C", "Humidity",
    "Rainfall_mm", "Sunlight_Hours", "Wind_Speed_kmh",
    "Field_Area_hectare", "Previous_Irrigation_mm",
]
CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]

# ── Pseudolabelling ───────────────────────────────────────────
PSEUDO_THRESH_BY_CLASS = {
    0: 0.97,   # Low   — strict
    1: 0.80,   # Medium — loose
    2: 0.97,   # High  — strict
}
PSEUDO_WEIGHT = 0.7

# ── Data config logged to W&B ─────────────────────────────────
DATA_CONFIG = {
    "fe_version"          : FE_VERSION,
    "augmentation_method" : "none",
    "use_original_data"   : True,
    "pseudo_label_version": "v2_asymmetric_thresh",
}

# ── XGBoost hyperparameters (Optuna-tuned) ────────────────────
XGB_PARAMS = dict(
    max_depth            = 4,
    learning_rate        = 0.030495387759654796,
    min_child_weight     = 2.333941903991847,
    subsample            = 0.9766412297733108,
    colsample_bytree     = 0.535324419516146,
    gamma                = 4.258489082295074,
    reg_alpha            = 4.082875850185249e-08,
    reg_lambda           = 0.00013528868091784412,
    objective            = "multi:softprob",
    num_class            = N_CLASSES,
    tree_method          = "hist",
    device               = "cuda",
    enable_categorical   = True,
    eval_metric          = "mlogloss",
    seed                 = 42,
    nthread              = -1,
    verbosity            = 0,
)

print(f"{'='*60}")
print(f"  {RUN_NAME}")
print(f"{'='*60}")
print(f"  FE version   : {FE_VERSION}")
print(f"  Device       : {XGB_PARAMS['device']}")
print(f"  Seeds        : {SEEDS}")
print(f"  N_FOLDS      : {N_FOLDS}")
print(f"  Total CV     : {len(SEEDS) * N_FOLDS} folds")
print(f"  learning_rate: {XGB_PARAMS['learning_rate']:.6f}")
print(f"{'='*60}")

# ============================================================
# SECTION 1 — W&B HELPER
# ============================================================
class WandbLogger:
    """Thin wrapper — W&B failures never crash training."""
    def __init__(self, enabled=True):
        self.enabled = enabled
        self.run     = None

    def init(self, config):
        if not self.enabled:
            return
        try:
            api_key = userdata.get("WB_TOKEN")
            wandb.login(key=api_key, relogin=True)
            self.run = wandb.init(
                project = WANDB_PROJECT,
                entity  = WANDB_ENTITY,
                name    = TAG,
                config  = config,
                tags    = ["xgboost", "v22", "ps-s6e4"],
            )
            print(f"  W&B run: {self.run.url}")
        except Exception as e:
            print(f"  [W&B] init failed: {e}")
            self.enabled = False

    def log(self, metrics, step=None):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.log(metrics, step=step) if step is not None \
                else wandb.log(metrics)
        except Exception as e:
            print(f"  [W&B] log failed: {e}")

    def log_confusion_matrix(self, y_true, y_pred, title):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.log({
                title: wandb.plot.confusion_matrix(
                    probs=None,
                    y_true=y_true.tolist(),
                    preds=y_pred.tolist(),
                    class_names=CLASS_NAMES,
                )
            })
        except Exception as e:
            print(f"  [W&B] confusion matrix failed: {e}")

    def log_artifact(self, local_path, artifact_name,
                     artifact_type, description=""):
        if not self.enabled or self.run is None:
            return
        try:
            art = wandb.Artifact(
                name=artifact_name, type=artifact_type,
                description=description,
            )
            art.add_file(local_path)
            self.run.log_artifact(art)
            print(f"  [W&B] artifact logged: {artifact_name}")
        except Exception as e:
            print(f"  [W&B] artifact failed: {e}")

    def summary(self, metrics):
        if not self.enabled or self.run is None:
            return
        try:
            for k, v in metrics.items():
                wandb.run.summary[k] = v
        except Exception as e:
            print(f"  [W&B] summary failed: {e}")

    def finish(self):
        if not self.enabled or self.run is None:
            return
        try:
            wandb.finish()
        except Exception as e:
            print(f"  [W&B] finish failed: {e}")


wb = WandbLogger(enabled=WANDB_ENABLED)

# ============================================================
# SECTION 2 — TELEGRAM NOTIFIER
# ============================================================
class TelegramNotifier:
    def __init__(self, bot_token=TELEGRAM_BOT_TOKEN,
                 chat_id=TELEGRAM_CHAT_ID,
                 enabled=TELEGRAM_ENABLED, run_name=RUN_NAME):
        self.bot_token  = bot_token
        self.chat_id    = chat_id
        self.enabled    = enabled
        self.run_name   = run_name
        self._start     = None
        self._hb_stop   = threading.Event()
        self._hb_thread = None

    def send(self, message, silent=False):
        if not self.enabled:
            return True
        try:
            url     = f"https://api.telegram.org/bot{self.bot_token}/sendMessage"
            payload = json.dumps({
                "chat_id"              : self.chat_id,
                "text"                 : message,
                "disable_notification" : silent,
            }).encode("utf-8")
            req = urllib.request.Request(
                url, data=payload,
                headers={"Content-Type": "application/json"},
            )
            urllib.request.urlopen(req, timeout=10)
            return True
        except Exception as e:
            print(f"[TELEGRAM] {e}")
            return False

    def start_timer(self):
        self._start = time.time()
        return self

    def elapsed(self):
        if self._start is None:
            return "unknown"
        s = int(time.time() - self._start)
        h, r = divmod(s, 3600)
        m, s = divmod(r, 60)
        return f"{h}h {m}m {s}s" if h else f"{m}m {s}s"

    def heartbeat(self, interval_minutes=20):
        if not self.enabled:
            return self
        if self._hb_thread is not None:
            self._hb_stop.set()
        self._hb_stop = threading.Event()
        def _loop():
            count = 0
            while not self._hb_stop.wait(interval_minutes * 60):
                count += 1
                self.send(
                    f"[{self.run_name}] running | "
                    f"{self.elapsed()} | hb#{count}",
                    silent=True,
                )
        self._hb_thread = threading.Thread(target=_loop, daemon=True)
        self._hb_thread.start()
        return self

    def stop_heartbeat(self):
        if self._hb_stop:
            self._hb_stop.set()

    def notify_seed(self, seed_idx, n_seeds, seed, seed_ba, silent=True):
        bar = "X" * seed_idx + "." * (n_seeds - seed_idx)
        self.send(
            f"[{self.run_name}] Seed {seed_idx}/{n_seeds} [{bar}]\n"
            f"  Seed: {seed} | OOF BA: {seed_ba:.6f} | {self.elapsed()}",
            silent=silent,
        )

    def success(self, oof_score, extra=""):
        self.stop_heartbeat()
        msg = (f"[{self.run_name}] Complete\n"
               f"  OOF BA : {oof_score:.6f}\n"
               f"  Runtime: {self.elapsed()}")
        if extra:
            msg += f"\n  {extra}"
        self.send(msg)

    def failure(self, exc=None, context=""):
        self.stop_heartbeat()
        tb = traceback.format_exc() if exc else ""
        if len(tb) > 800:
            tb = "..." + tb[-800:]
        lines = [f"[{self.run_name}] FAILED", f"  Elapsed: {self.elapsed()}"]
        if context:
            lines.append(f"  Context: {context}")
        if exc:
            lines.append(f"  {type(exc).__name__}: {exc}")
        if tb:
            lines.append(tb)
        self.send("\n".join(lines))

    @contextmanager
    def run_context(self, context=""):
        try:
            yield
        except Exception as e:
            self.failure(exc=e, context=context)
            raise


notifier = TelegramNotifier()

# ============================================================
# SECTION 3 — REPRODUCIBILITY
# ============================================================
def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

seed_everything()

# ============================================================
# SECTION 4 — CUSTOM EVAL METRIC
# ============================================================
def feval_bal_acc(preds, dmatrix):
    labels = dmatrix.get_label().astype(int)
    probs  = preds.reshape(-1, N_CLASSES)
    y_pred = probs.argmax(axis=1)
    score  = balanced_accuracy_score(labels, y_pred)
    return "bal_acc", score

# ============================================================
# SECTION 5 — TARGET ENCODING HELPER
# ============================================================
def apply_te(X_tr_p, y_tr, X_va_p, X_te_p, cols, seed=42):
    enc = TargetEncoder(
        target_type="multiclass",
        cv=N_FOLDS,
        random_state=seed,
    )
    tr_enc = pd.DataFrame(
        enc.fit_transform(X_tr_p[cols], y_tr)
    ).reset_index(drop=True)
    va_enc = pd.DataFrame(
        enc.transform(X_va_p[cols])
    ).reset_index(drop=True)
    te_enc = pd.DataFrame(
        enc.transform(X_te_p[cols])
    ).reset_index(drop=True)
    return tr_enc, va_enc, te_enc

# ============================================================
# SECTION 6 — BIAS TUNING
# ============================================================
def tune_logit_bias(oof_probs, y_true):
    def get_preds(probs, bias):
        adj = logit(np.clip(probs, 1e-15, 1-1e-15)) + bias
        return np.argmax(adj, axis=1)

    best_bias   = np.zeros(3)
    best_score  = balanced_accuracy_score(y_true, oof_probs.argmax(1))
    raw_score   = best_score
    opt_history = [best_score]

    for step in [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002]:
        improved = True
        while improved:
            improved = False
            for ci in range(3):
                for d in [1, -1]:
                    trial      = best_bias.copy()
                    trial[ci] += d * step
                    s = balanced_accuracy_score(
                        y_true, get_preds(oof_probs, trial)
                    )
                    if s > best_score + 1e-9:
                        best_score, best_bias, improved = s, trial, True
                        opt_history.append(best_score)

    print(f"  Bias raw  : {raw_score:.6f}")
    print(f"  Bias tuned: {best_score:.6f} (+{best_score - raw_score:.6f})")
    print(f"  Biases    : Low={best_bias[0]:.4f} "
          f"Med={best_bias[1]:.4f} High={best_bias[2]:.4f}")
    return best_bias, best_score, opt_history


def apply_bias(probs, bias):
    log_p = logit(np.clip(probs, 1e-15, 1-1e-15)) + bias
    exp_p = np.exp(log_p)
    return (exp_p / exp_p.sum(axis=1, keepdims=True)).astype(np.float32)

# ============================================================
# SECTION 7 — MAIN PIPELINE
# ============================================================
if __name__ == "__main__":
    notifier.start_timer().heartbeat(interval_minutes=20)

    # ── W&B init ─────────────────────────────────────────────
    wb_config = {
        **DATA_CONFIG,
        **XGB_PARAMS,
        "seeds"               : SEEDS,
        "n_folds"             : N_FOLDS,
        "total_cv_folds"      : len(SEEDS) * N_FOLDS,
        "pseudo_thresh_low"   : PSEUDO_THRESH_BY_CLASS[0],
        "pseudo_thresh_medium": PSEUDO_THRESH_BY_CLASS[1],
        "pseudo_thresh_high"  : PSEUDO_THRESH_BY_CLASS[2],
        "pseudo_weight"       : PSEUDO_WEIGHT,
        "tag"                 : TAG,
        "version"             : "v22",
    }
    wb.init(config=wb_config)

    notifier.send(
        f"[{RUN_NAME}] Starting\n"
        f"  FE: {FE_VERSION}\n"
        f"  Seeds: {SEEDS}\n"
        f"  {len(SEEDS)} × {N_FOLDS}-fold = {len(SEEDS)*N_FOLDS} folds",
        silent=True,
    )

    try:
        # ── 1. Load pre-engineered parquet ────────────────────
        print(f"\n[1] Loading pre-engineered features (FE {FE_VERSION})...")

        # Verify metadata
        with open(FE_META_PATH) as f:
            meta = json.load(f)
        assert meta["fe_version"] == FE_VERSION, \
            f"FE version mismatch: {meta['fe_version']} != {FE_VERSION}"

        # Checksum verification
        actual_ck = hashlib.md5(
            open(TRAIN_PARQUET, "rb").read()
        ).hexdigest()
        assert actual_ck == meta["train_checksum"], \
            f"Train parquet checksum mismatch.\n" \
            f"  Expected : {meta['train_checksum']}\n" \
            f"  Got      : {actual_ck}\n" \
            f"  Re-run feature_engineering_v21.py to regenerate."

        train_eng = pd.read_parquet(TRAIN_PARQUET)
        test_eng  = pd.read_parquet(TEST_PARQUET)

        # Recover column lists from metadata
        NUMS         = meta["num_cols"]
        CATS         = meta["cat_cols"]
        NEW_FEATURES = meta["new_features"]
        n_competition= meta["n_competition"]

        print(f"  Train: {train_eng.shape}  Test: {test_eng.shape}")
        print(f"  FE {FE_VERSION} loaded and verified ✅")
        print(f"  Competition rows: {n_competition:,}")
        print(f"  New features    : {len(NEW_FEATURES)}")

        # Log parquet as W&B artifact (Point 1)
        wb.log_artifact(
            local_path    = TRAIN_PARQUET,
            artifact_name = f"train_engineered_{FE_VERSION}",
            artifact_type = "dataset",
            description   = f"Pre-engineered train parquet FE {FE_VERSION}",
        )

        # ── 2. Load test IDs ──────────────────────────────────
        # Original data is already baked into the parquet by
        # feature_engineering_v21.py — no need to reload it here.
        test_raw = pd.read_csv(TEST_PATH)
        test_ids = test_raw["id"].copy()

        n_orig = len(train_eng) - n_competition
        print(f"\n  Test rows    : {len(test_raw):,}")
        print(f"  Original rows: {n_orig:,} (already in parquet)")

        # ── 3. Load hard example indices ─────────────────────
        hard_mask = None
        if os.path.exists(HARD_IDX_PATH):
            hard_idx  = np.load(HARD_IDX_PATH)
            hard_mask = np.zeros(n_competition, dtype=bool)
            hard_mask[hard_idx] = True
            print(f"\n  Hard examples: {hard_mask.sum():,} rows loaded ✅")
        else:
            print(f"\n  Hard examples: not found — skipping hard-row analysis")

        # ── 4. Label encode categoricals ─────────────────────
        print(f"\n[2] Label encoding categoricals...")
        for col in CATS:
            le = LabelEncoder()
            combined = pd.concat([
                train_eng[col].astype(str),
                test_eng[col].astype(str),
            ])
            le.fit(combined)
            train_eng[col] = le.transform(
                train_eng[col].astype(str)
            ).astype("int32")
            test_eng[col]  = le.transform(
                test_eng[col].astype(str)
            ).astype("int32")

        # ── 5. Pairwise interactions (CAT×CAT + CAT×NUM only) ─
        print(f"\n[3] Building pairwise interactions "
              f"(CAT×CAT + CAT×NUM, skipping NUM×NUM)...")
        ALL_COLS   = NUMS + CATS
        TE_COLUMNS = []
        n_tr       = len(train_eng)
        total_len  = n_tr + len(test_eng)

        for cols in tqdm(list(combinations(ALL_COLS, 2))):
            # Skip NUM×NUM — already captured by domain features in parquet
            if cols[0] in NUMS and cols[1] in NUMS:
                continue
            name     = f"{cols[0]}|{cols[1]}"
            tr_s     = (train_eng[cols[0]].astype(str) + "_"
                        + train_eng[cols[1]].astype(str))
            te_s     = (test_eng[cols[0]].astype(str) + "_"
                        + test_eng[cols[1]].astype(str))
            combined = pd.concat([tr_s, te_s], ignore_index=True)
            codes, _ = combined.factorize()
            if pd.Series(codes).nunique() > total_len // 2:
                continue
            train_eng[name] = codes[:n_tr].astype("int32")
            test_eng[name]  = codes[n_tr:].astype("int32")
            TE_COLUMNS.append(name)

        print(f"  Kept {len(TE_COLUMNS)} interaction columns")
        wb.log({"n_te_columns": len(TE_COLUMNS)})

        # ── 6. Prepare base feature matrices ─────────────────
        print(f"\n[4] Preparing feature matrices...")

        # The parquet already contains competition rows + original rows
        # (feature_engineering_v21.py concatenated them before saving).
        # train_eng.shape[0] == n_competition + n_orig
        # No manual orig_df engineering needed — it's already in the parquet.
        train_full_eng = train_eng.copy()

        y_full = train_full_eng[TARGET].values.astype(np.int32)

        DROP_COLS     = {"id", TARGET}
        BASE_FEATURES = [
            c for c in train_full_eng.columns
            if c not in DROP_COLS
            and c not in TE_COLUMNS
            and train_full_eng[c].dtype != object
        ]

        medians = train_full_eng[BASE_FEATURES].median()

        X_base  = (train_full_eng[BASE_FEATURES]
                   .fillna(medians)
                   .astype("float32"))
        X_tbase = (test_eng[[c for c in BASE_FEATURES
                              if c in test_eng.columns]]
                   .fillna(medians)
                   .astype("float32"))
        for c in BASE_FEATURES:
            if c not in X_tbase.columns:
                X_tbase[c] = 0.0
        X_tbase = X_tbase[BASE_FEATURES]

        X_pair  = train_full_eng[TE_COLUMNS].astype("int32").copy()
        X_tpair = test_eng[TE_COLUMNS].astype("int32").copy()

        # Free large frames — no longer needed
        del train_eng, test_eng, train_full_eng
        gc.collect()

        sample_weights = compute_sample_weight("balanced", y_full)

        print(f"  Base features : {len(BASE_FEATURES)}")
        print(f"  TE columns    : {len(TE_COLUMNS)}")
        print(f"  Total rows    : {len(y_full):,}  "
              f"(comp: {n_competition:,} + orig: {len(y_full)-n_competition:,})")
        print(f"  Class dist    : "
              f"{dict(zip(*np.unique(y_full, return_counts=True)))}")

        # Zero-variance guard on hard rows
        if hard_mask is not None:
            print(f"\n  Checking zero-variance on hard rows...")
            hard_base  = X_base.iloc[:n_competition].iloc[hard_mask]
            zero_var   = [
                BASE_FEATURES[i]
                for i, v in enumerate(hard_base.var(axis=0).values)
                if v < 1e-10
            ]
            if zero_var:
                raise ValueError(
                    f"❌ ZERO-VARIANCE features on hard rows:\n"
                    + "\n".join(f"   - {c}" for c in zero_var)
                    + "\n\nFix in feature_engineering_v21.py before training."
                )
            print(f"  ✅ Zero-variance check passed "
                  f"({hard_mask.sum():,} hard rows)")

        wb.log({
            "n_base_features": len(BASE_FEATURES),
            "n_competition"  : n_competition,
            "n_train_full"   : len(y_full),
        })

        notifier.send(
            f"[{RUN_NAME}] Data ready\n"
            f"  Base: {len(BASE_FEATURES)} | TE: {len(TE_COLUMNS)}\n"
            f"  Elapsed: {notifier.elapsed()}",
            silent=True,
        )

        # ── 7. Multi-seed CV ──────────────────────────────────
        print(f"\n[5] Multi-seed CV "
              f"({len(SEEDS)} seeds × {N_FOLDS} folds)...")

        oof_accum       = np.zeros((len(y_full),  N_CLASSES), dtype=np.float64)
        test_accum      = np.zeros((len(X_tbase), N_CLASSES), dtype=np.float64)
        seed_oof_scores = []
        all_best_iters  = []
        total_start     = time.time()
        # Monotonic counter — incremented BEFORE each wb.log() call
        # so W&B steps are always strictly 1, 2, ..., 25
        global_fold_num = 0

        for seed_idx, seed in enumerate(SEEDS):
            seed_start = time.time()
            print(f"\n{'─'*60}")
            print(f"  SEED {seed}  ({seed_idx+1}/{len(SEEDS)})")
            print(f"{'─'*60}")

            seed_everything(seed)
            skf = StratifiedKFold(
                n_splits=N_FOLDS, shuffle=True, random_state=seed
            )

            oof_seed  = np.zeros((len(y_full),  N_CLASSES), dtype=np.float64)
            test_seed = np.zeros((len(X_tbase), N_CLASSES), dtype=np.float64)
            seed_iters = []

            for fold, (tr_idx, val_idx) in enumerate(
                skf.split(X_base, y_full)
            ):
                fold_start = time.time()
                print(f"\n  Fold {fold+1}/{N_FOLDS} | Seed {seed}")

                with notifier.run_context(f"Seed {seed} Fold {fold+1}"):
                    # Target encoding on pairwise columns
                    tr_enc, va_enc, te_enc = apply_te(
                        X_pair.iloc[tr_idx], y_full[tr_idx],
                        X_pair.iloc[val_idx], X_tpair,
                        TE_COLUMNS, seed=seed,
                    )

                    # numpy hstack — avoids pd.concat copy overhead
                    Xt = np.hstack([
                        X_base.iloc[tr_idx].values,
                        tr_enc.values.astype("float32"),
                    ])
                    Xv = np.hstack([
                        X_base.iloc[val_idx].values,
                        va_enc.values.astype("float32"),
                    ])
                    Xe = np.hstack([
                        X_tbase.values,
                        te_enc.values.astype("float32"),
                    ])

                    dtrain = xgb.DMatrix(
                        Xt, label=y_full[tr_idx],
                        weight=sample_weights[tr_idx],
                        enable_categorical=True,
                    )
                    dval   = xgb.DMatrix(
                        Xv, label=y_full[val_idx],
                        enable_categorical=True,
                    )
                    dtest  = xgb.DMatrix(Xe, enable_categorical=True)

                    del Xt, Xv, Xe, tr_enc, va_enc, te_enc
                    gc.collect()

                    model = xgb.train(
                        params                = XGB_PARAMS,
                        dtrain                = dtrain,
                        num_boost_round       = 10000,
                        evals                 = [(dval, "val")],
                        custom_metric         = feval_bal_acc,
                        maximize              = True,
                        early_stopping_rounds = 300,
                        verbose_eval          = 200,
                    )

                    best_iter = model.best_iteration
                    seed_iters.append(best_iter)
                    all_best_iters.append(best_iter)

                    val_proba  = model.predict(dval).reshape(-1, N_CLASSES)
                    test_proba = model.predict(dtest).reshape(-1, N_CLASSES)

                    oof_seed[val_idx] += val_proba
                    test_seed         += test_proba

                    fold_ba  = balanced_accuracy_score(
                        y_full[val_idx], val_proba.argmax(axis=1)
                    )
                    elapsed  = int(time.time() - fold_start)
                    print(f"    BA={fold_ba:.5f} | iter={best_iter} | "
                          f"{elapsed//60}m {elapsed%60}s")

                    # W&B fold metrics — monotonic step (Point 2)
                    global_fold_num += 1
                    wb.log({
                        "fold_ba"       : fold_ba,
                        "fold_best_iter": best_iter,
                        "fold_elapsed_s": elapsed,
                        "seed"          : seed,
                        "fold"          : fold + 1,
                        "seed_idx"      : seed_idx + 1,
                        "global_fold"   : global_fold_num,
                    }, step=global_fold_num)

                    del model, dtrain, dval, dtest
                    gc.collect()

            test_seed /= N_FOLDS
            seed_ba    = balanced_accuracy_score(
                y_full, oof_seed.argmax(1)
            )
            seed_oof_scores.append(seed_ba)

            seed_elapsed = int(time.time() - seed_start)
            print(f"\n  Seed {seed} OOF BA: {seed_ba:.6f} | "
                  f"Avg iter: {int(np.mean(seed_iters))} | "
                  f"{seed_elapsed//60}m {seed_elapsed%60}s")

            # Seed-level metrics — no step (goes to summary panel)
            wb.log({
                f"seed_{seed}_oof_ba": seed_ba,
                "seed_oof_ba"        : seed_ba,
                "seed_avg_iter"      : int(np.mean(seed_iters)),
                "seed_elapsed_s"     : seed_elapsed,
            })

            notifier.notify_seed(seed_idx+1, len(SEEDS), seed, seed_ba)

            oof_accum  += oof_seed
            test_accum += test_seed

        oof_accum  /= len(SEEDS)
        test_accum /= len(SEEDS)
        avg_iter    = int(np.mean(all_best_iters))

        total_elapsed = int(time.time() - total_start)
        print(f"\n{'='*60}")
        print(f"Multi-seed CV complete | "
              f"{total_elapsed//3600}h "
              f"{(total_elapsed%3600)//60}m {total_elapsed%60}s")
        print(f"Per-seed OOF BAs : "
              f"{[round(s, 5) for s in seed_oof_scores]}")
        print(f"Mean seed BA     : {np.mean(seed_oof_scores):.6f} "
              f"± {np.std(seed_oof_scores):.6f}")

        # ── 8. Bias tuning ────────────────────────────────────
        print(f"\n[6] Bias tuning on averaged OOF...")
        oof_comp_avg = oof_accum[:n_competition].astype(np.float32)
        y_comp       = y_full[:n_competition]

        raw_avg_ba = balanced_accuracy_score(
            y_comp, oof_comp_avg.argmax(1)
        )
        print(f"  Averaged OOF raw BA (comp rows): {raw_avg_ba:.6f}")

        best_bias, tuned_ba, opt_history = tune_logit_bias(
            oof_comp_avg, y_comp
        )

        oof_calibrated  = apply_bias(
            oof_accum[:n_competition].astype(np.float32), best_bias
        )
        test_calibrated = apply_bias(
            test_accum.astype(np.float32), best_bias
        )

        # W&B global OOF metrics (Point 2)
        per_class_ba = {}
        for cls in range(3):
            mask = y_comp == cls
            per_class_ba[f"oof_ba_{CLASS_NAMES[cls].lower()}"] = float(
                (oof_calibrated[mask].argmax(axis=1) == cls).mean()
            )

        wb.log({
            "oof_ba_raw"      : raw_avg_ba,
            "oof_ba_biased"   : tuned_ba,
            "bias_correction" : tuned_ba - raw_avg_ba,
            "avg_best_iter"   : avg_iter,
            "mean_seed_ba"    : float(np.mean(seed_oof_scores)),
            "std_seed_ba"     : float(np.std(seed_oof_scores)),
            "bias_low"        : float(best_bias[0]),
            "bias_medium"     : float(best_bias[1]),
            "bias_high"       : float(best_bias[2]),
            **per_class_ba,
        })

        print(f"\n  Per-class OOF BA (post-bias):")
        for cls in range(3):
            print(f"    {CLASS_NAMES[cls]:<8}: "
                  f"{per_class_ba[f'oof_ba_{CLASS_NAMES[cls].lower()}']:.5f}")

        # W&B full OOF confusion matrix (Point 3)
        wb.log_confusion_matrix(
            y_true=y_comp,
            y_pred=oof_calibrated.argmax(axis=1),
            title="oof_confusion_matrix",
        )

        # W&B hard-example analysis (Point 3)
        if hard_mask is not None:
            hard_oof_raw = oof_accum[:n_competition][hard_mask].astype(np.float32)
            hard_true    = y_comp[hard_mask]

            hard_ba_pre  = balanced_accuracy_score(
                hard_true, hard_oof_raw.argmax(axis=1)
            )
            hard_ba_post = balanced_accuracy_score(
                hard_true, oof_calibrated[hard_mask].argmax(axis=1)
            )
            print(f"\n  Hard-example OOF BA pre-bias : {hard_ba_pre:.5f}")
            print(f"  Hard-example OOF BA post-bias: {hard_ba_post:.5f}")

            wb.log({
                "hard_oof_ba_pre_bias" : hard_ba_pre,
                "hard_oof_ba_post_bias": hard_ba_post,
            })
            wb.log_confusion_matrix(
                y_true=hard_true,
                y_pred=oof_calibrated[hard_mask].argmax(axis=1),
                title="hard_example_confusion_matrix",
            )
            hard_per_class = {}
            for cls in range(3):
                m = hard_true == cls
                if m.sum() > 0:
                    hard_per_class[f"hard_ba_{CLASS_NAMES[cls].lower()}"] = float(
                        (oof_calibrated[hard_mask][m].argmax(axis=1) == cls).mean()
                    )
            wb.log(hard_per_class)

        # ── 9. Diagnostic plots ───────────────────────────────
        print(f"\n[7] Generating diagnostic plots...")

        fig1, ax1 = plt.subplots(figsize=(9, 4))
        bars = ax1.bar(
            [str(s) for s in SEEDS], seed_oof_scores,
            color="steelblue", alpha=0.8, edgecolor="black",
        )
        ax1.axhline(
            np.mean(seed_oof_scores), color="red", linestyle="--",
            label=f"Mean: {np.mean(seed_oof_scores):.5f}",
        )
        for bar, val in zip(bars, seed_oof_scores):
            ax1.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.0001,
                f"{val:.5f}", ha="center", va="bottom", fontsize=9,
            )
        ax1.set_title(f"Per-Seed OOF BA ({RUN_NAME})", fontweight="bold")
        ax1.set_xlabel("Seed"); ax1.set_ylabel("Balanced Accuracy")
        ax1.legend(); plt.tight_layout()
        wb.log({"chart_seed_oof_ba": wandb.Image(fig1)})
        plt.show(); plt.close(fig1)

        fig2, ax2 = plt.subplots(figsize=(10, 4))
        ax2.plot(
            range(len(opt_history)), opt_history,
            color="darkorange", marker="o", markersize=4, linewidth=1.5,
        )
        ax2.axhline(
            raw_avg_ba, color="gray", linestyle="--",
            label="Raw averaged OOF BA",
        )
        ax2.set_title(
            f"Bias Tuning: {raw_avg_ba:.5f} → {tuned_ba:.5f} "
            f"(+{tuned_ba - raw_avg_ba:.5f})",
            fontweight="bold",
        )
        ax2.set_xlabel("Step"); ax2.set_ylabel("Balanced Accuracy")
        ax2.legend(); plt.tight_layout()
        wb.log({"chart_bias_tuning": wandb.Image(fig2)})
        plt.show(); plt.close(fig2)

        fig3, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.kdeplot(
            oof_calibrated.max(1), ax=axes[0], label="OOF",
            fill=True, color="steelblue", alpha=0.5,
        )
        sns.kdeplot(
            test_calibrated.max(1), ax=axes[0], label="Test",
            fill=True, color="darkorange", alpha=0.3,
        )
        axes[0].set_title("Confidence Distribution"); axes[0].legend()
        oof_dist  = pd.Series(
            [INV_TARGET_MAPPING[p] for p in oof_calibrated.argmax(1)]
        ).value_counts(normalize=True).sort_index()
        test_dist = pd.Series(
            [INV_TARGET_MAPPING[p] for p in test_calibrated.argmax(1)]
        ).value_counts(normalize=True).sort_index()
        x = np.arange(3)
        axes[1].bar(x-0.2, oof_dist.values,  0.4,
                    label="OOF",  color="steelblue",  alpha=0.7)
        axes[1].bar(x+0.2, test_dist.values, 0.4,
                    label="Test", color="darkorange", alpha=0.7)
        axes[1].set_title("Class Distribution")
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(oof_dist.index)
        axes[1].legend()
        plt.tight_layout()
        wb.log({"chart_confidence_dist": wandb.Image(fig3)})
        plt.show(); plt.close(fig3)

        # ── 10. Pseudolabelling ───────────────────────────────
        print(f"\n[8] Pseudolabeling (asymmetric thresholds)...")
        print(f"  Thresholds: Low={PSEUDO_THRESH_BY_CLASS[0]} | "
              f"Medium={PSEUDO_THRESH_BY_CLASS[1]} | "
              f"High={PSEUDO_THRESH_BY_CLASS[2]}")
        print(f"  Pseudo row weight: {PSEUDO_WEIGHT}")

        pseudo_labels_all = test_calibrated.argmax(axis=1)
        pseudo_conf_all   = test_calibrated.max(axis=1)
        pseudo_mask       = np.array([
            pseudo_conf_all[i] >= PSEUDO_THRESH_BY_CLASS[pseudo_labels_all[i]]
            for i in range(len(test_calibrated))
        ])
        pseudo_labels = pseudo_labels_all[pseudo_mask]

        print(f"  High-confidence test samples: {pseudo_mask.sum():,}")
        pseudo_class_counts = {}
        for cls in range(3):
            n = (pseudo_labels == cls).sum()
            pseudo_class_counts[f"pseudo_n_{CLASS_NAMES[cls].lower()}"] = int(n)
            print(f"    {INV_TARGET_MAPPING[cls]}: {n:,}")

        wb.log({
            "pseudo_n_total": int(pseudo_mask.sum()),
            **pseudo_class_counts,
        })

        if pseudo_mask.sum() > 500:
            tr_enc_full, _, te_enc_full = apply_te(
                X_pair.iloc[:n_competition],
                y_full[:n_competition],
                X_pair.iloc[:1],
                X_tpair,
                TE_COLUMNS, seed=SEEDS[0],
            )

            X_full   = np.hstack([
                X_base.iloc[:n_competition].values,
                tr_enc_full.values.astype("float32"),
            ])
            X_test_f = np.hstack([
                X_tbase.values,
                te_enc_full.values.astype("float32"),
            ])

            X_aug = np.vstack([X_full, X_test_f[pseudo_mask]])
            y_aug = np.concatenate(
                [y_full[:n_competition], pseudo_labels]
            )

            w_real   = compute_sample_weight(
                "balanced", y_full[:n_competition]
            )
            w_pseudo = np.full(pseudo_mask.sum(), PSEUDO_WEIGHT)
            w_aug    = np.concatenate([w_real, w_pseudo])

            print(f"  Retraining on {len(X_aug):,} rows "
                  f"({pseudo_mask.sum():,} pseudo) "
                  f"for {avg_iter} rounds...")

            dtrain_ps = xgb.DMatrix(
                X_aug, label=y_aug, weight=w_aug,
                enable_categorical=True,
            )
            dtest_ps  = xgb.DMatrix(X_test_f, enable_categorical=True)
            model_ps  = xgb.train(
                params=XGB_PARAMS,
                dtrain=dtrain_ps,
                num_boost_round=avg_iter,
                verbose_eval=False,
            )

            final_test_probs = model_ps.predict(dtest_ps).reshape(
                -1, N_CLASSES
            ).astype(np.float32)
            final_test_probs = apply_bias(final_test_probs, best_bias)

            print("  Pseudolabeling complete.")
            del model_ps, dtrain_ps, dtest_ps
            gc.collect()
        else:
            final_test_probs = test_calibrated
            print("  Skipped — insufficient high-confidence samples.")

        # ── 11. Save + W&B Artifacts ──────────────────────────
        print(f"\n[9] Saving...")

        oof_path  = f"{OOF_DIR}oof_{TAG}.npy"
        pred_path = f"{OOF_DIR}pred_{TAG}.npy"

        np.save(oof_path,
                oof_calibrated.astype(np.float32))
        np.save(pred_path,
                final_test_probs.astype(np.float32))
        np.save(f"{OOF_DIR}oof_{TAG}_biased.npy",
                oof_calibrated.astype(np.float32))
        np.save(f"{OOF_DIR}pred_{TAG}_biased.npy",
                final_test_probs.astype(np.float32))

        print(f"  oof_{TAG}.npy   {oof_calibrated.shape}")
        print(f"  pred_{TAG}.npy  {final_test_probs.shape}")

        assert oof_calibrated.shape   == (n_competition, N_CLASSES), \
            f"OOF shape mismatch: {oof_calibrated.shape}"
        assert final_test_probs.shape == (len(test_raw),  N_CLASSES), \
            f"Pred shape mismatch: {final_test_probs.shape}"
        print("  Shape assertions passed ✅")

        # Log OOF + pred as W&B Artifacts (Point 1)
        wb.log_artifact(
            local_path    = oof_path,
            artifact_name = f"oof_{TAG}",
            artifact_type = "model_output",
            description   = f"OOF probs | biased BA={tuned_ba:.6f}",
        )
        wb.log_artifact(
            local_path    = pred_path,
            artifact_name = f"pred_{TAG}",
            artifact_type = "model_output",
            description   = f"Test probs | biased BA={tuned_ba:.6f}",
        )

        # ── 12. Submission ────────────────────────────────────
        sub = pd.DataFrame({
            "id"  : test_ids,
            TARGET: [INV_TARGET_MAPPING[p]
                     for p in final_test_probs.argmax(axis=1)],
        })
        sub_path = f"{SUB_DIR}submission_{TAG}.csv"
        sub.to_csv(sub_path, index=False)
        print(f"\n  Submission: {sub_path}")
        print(sub[TARGET].value_counts().to_string())

        wb.log({
            f"submission_n_{k.lower()}": int(v)
            for k, v in sub[TARGET].value_counts().items()
        })

        # ── 13. W&B summary + final print ─────────────────────
        wb.summary({
            "oof_ba_raw"     : raw_avg_ba,
            "oof_ba_biased"  : tuned_ba,
            "bias_correction": tuned_ba - raw_avg_ba,
            "mean_seed_ba"   : float(np.mean(seed_oof_scores)),
            "avg_best_iter"  : avg_iter,
            "fe_version"     : FE_VERSION,
            "tag"            : TAG,
        })

        print(f"\n{'='*60}")
        print(f"{RUN_NAME} SUMMARY")
        print(f"{'='*60}")
        print(f"  FE version     : {FE_VERSION}")
        print(f"  Seeds          : {SEEDS}")
        print(f"  Per-seed BAs   : "
              f"{[round(s, 5) for s in seed_oof_scores]}")
        print(f"  Mean seed BA   : {np.mean(seed_oof_scores):.6f} "
              f"± {np.std(seed_oof_scores):.6f}")
        print(f"  Averaged raw BA: {raw_avg_ba:.6f}")
        print(f"  Biased BA      : {tuned_ba:.6f} "
              f"(+{tuned_ba - raw_avg_ba:.6f})")
        print(f"  Biases         : {np.round(best_bias, 4)}")
        print(f"  Avg best iter  : {avg_iter}")
        print(f"  Runtime        : {notifier.elapsed()}")
        print(f"{'='*60}")

        notifier.success(
            oof_score=tuned_ba,
            extra=(
                f"raw={raw_avg_ba:.5f} | "
                f"FE={FE_VERSION} | "
                f"avg_iter={avg_iter}"
            ),
        )

    except Exception as e:
        notifier.failure(exc=e, context=f"Main {RUN_NAME}")
        raise

    finally:
        # Always close W&B — even on crash
        wb.finish()

  XGB v22
  FE version   : v21
  Device       : cuda
  Seeds        : [42, 123, 2024, 7, 314]
  N_FOLDS      : 5
  Total CV     : 25 folds
  learning_rate: 0.030495


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


  W&B run: https://wandb.ai/wblackstone-twilight-signals/ps-s6e4-irrigation/runs/10n1xv71

[1] Loading pre-engineered features (FE v21)...
  Train: (640000, 111)  Test: (270000, 110)
  FE v21 loaded and verified ✅
  Competition rows: 630,000
  New features    : 90
  [W&B] artifact logged: train_engineered_v21

  Test rows    : 270,000
  Original rows: 10,000 (already in parquet)

  Hard examples: 7,377 rows loaded ✅

[2] Label encoding categoricals...

[3] Building pairwise interactions (CAT×CAT + CAT×NUM, skipping NUM×NUM)...


100%|██████████| 171/171 [01:28<00:00,  1.93it/s]


  Kept 116 interaction columns

[4] Preparing feature matrices...
  Base features : 109
  TE columns    : 116
  Total rows    : 640,000  (comp: 630,000 + orig: 10,000)
  Class dist    : {np.int32(0): np.int64(375781), np.int32(1): np.int64(242874), np.int32(2): np.int64(21345)}

  Checking zero-variance on hard rows...
  ✅ Zero-variance check passed (7,377 hard rows)

[5] Multi-seed CV (5 seeds × 5 folds)...

────────────────────────────────────────────────────────────
  SEED 42  (1/5)
────────────────────────────────────────────────────────────

  Fold 1/5 | Seed 42
[0]	val-mlogloss:1.05621	val-bal_acc:0.97323
[200]	val-mlogloss:0.06772	val-bal_acc:0.97725
[400]	val-mlogloss:0.05938	val-bal_acc:0.97862
[600]	val-mlogloss:0.05626	val-bal_acc:0.97905
[800]	val-mlogloss:0.05443	val-bal_acc:0.97921
[1000]	val-mlogloss:0.05332	val-bal_acc:0.97924
[1162]	val-mlogloss:0.05278	val-bal_acc:0.97895
    BA=0.97895 | iter=862 | 2m 13s

  Fold 2/5 | Seed 42


wandb: WARNING Tried to log to step 1 that is less than the current step 2. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


[0]	val-mlogloss:1.05620	val-bal_acc:0.97406
[200]	val-mlogloss:0.06677	val-bal_acc:0.97742
[400]	val-mlogloss:0.05850	val-bal_acc:0.97851
[600]	val-mlogloss:0.05556	val-bal_acc:0.97900
[800]	val-mlogloss:0.05374	val-bal_acc:0.97869
[929]	val-mlogloss:0.05302	val-bal_acc:0.97834
    BA=0.97834 | iter=629 | 2m 7s

  Fold 3/5 | Seed 42
[0]	val-mlogloss:1.05619	val-bal_acc:0.97215
[200]	val-mlogloss:0.06664	val-bal_acc:0.97652
[400]	val-mlogloss:0.05824	val-bal_acc:0.97844
[600]	val-mlogloss:0.05531	val-bal_acc:0.97899
[800]	val-mlogloss:0.05359	val-bal_acc:0.97914
[1000]	val-mlogloss:0.05255	val-bal_acc:0.97903
[1112]	val-mlogloss:0.05222	val-bal_acc:0.97907
    BA=0.97907 | iter=812 | 2m 14s

  Fold 4/5 | Seed 42
[0]	val-mlogloss:1.05620	val-bal_acc:0.97256
[200]	val-mlogloss:0.06705	val-bal_acc:0.97645
[400]	val-mlogloss:0.05881	val-bal_acc:0.97735
[600]	val-mlogloss:0.05582	val-bal_acc:0.97799
[800]	val-mlogloss:0.05403	val-bal_acc:0.97815
[1000]	val-mlogloss:0.05296	val-bal_acc:0.978